# Migrasjons-tidsserie per kommune

Inn- og utflyttinger per kommune per måned fra `silver/folkeregister/municipality_migration`.

## 1. Last og inspiser

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName('migration').getOrCreate()

df = spark.read.format('delta').load('s3a://silver/folkeregister/municipality_migration')
df.show(5)

## 2. Kommuner med høyest netto innflytting siste 6 måneder

In [ ]:
from datetime import datetime, timedelta
cutoff = (datetime.utcnow() - timedelta(days=180)).strftime('%Y-%m-%d')

(df.filter(F.col('period') >= cutoff)
  .groupBy('municipality_name')
  .agg(F.sum('netto').alias('netto_total'))
  .orderBy(F.desc('netto_total'))
  .show(15))

## 3. Plot tidsserien for én kommune

In [ ]:
kommune = 'Oslo'
ts = (df.filter(F.col('municipality_name') == kommune)
         .orderBy('period')
         .toPandas())

import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))
plt.plot(ts['period'], ts['innflyttinger'], label='Innflyttinger')
plt.plot(ts['period'], ts['utflyttinger'], label='Utflyttinger')
plt.title(f'Migrasjon i {kommune}')
plt.legend(); plt.xticks(rotation=45); plt.tight_layout()